# 알약 검출 실험: YOLO11s와 YOLO26s + EasyOCR

이 노트북에는 **서로 다른 YOLO 검출 모델을 학습하는 실험 2개**가 있습니다.
두 실험 모두 검출한 알약 영역을 잘라 사전학습 EasyOCR로 각인을 읽습니다.
**학습 대상은 YOLO이며, EasyOCR는 학습하지 않습니다.**

| 구간 | 실험 목적 | 학습하는 모델 | OCR | 대표 변수 |
|---|---|---|---|---|
| 공통 준비 | 경로·패키지·설정·train/val 데이터 준비 | 없음 | 없음 | `DATA_YAML`, `SEED`, `TRAIN_CONFIG` |
| 실험 1 | YOLO11s 알약 검출 학습 및 각인 추론 | `yolo11s.pt`에서 시작한 YOLO11s | 사전학습 EasyOCR | `model`, `detector`, `records` |
| 실험 2 | 같은 데이터 분할로 YOLO26s 검출 실험 | `yolo26s.pt`에서 시작한 YOLO26s | 사전학습 EasyOCR | `yolo26_model`, `yolo26_detector`, `yolo26_records` |

기본 설정은 **두 모델 모두 전체 학습 데이터·최대 20 epoch 본 학습**입니다.
배치 크기·증강 등 다른 설정도 다르므로 현재 수치만으로 모델 구조의 우열을 정하지 않습니다.
동일한 조건의 비교가 필요하면 학습량과 설정을 맞춰 별도로 실험하세요.

이 설명은 코드의 역할을 나타냅니다. 저장된 노트북에는 실행 출력이 없으므로,
실제 학습 완료 여부와 성능 수치는 실행한 Colab의 로그와 `best.pt`, 검증 결과로 확인하세요.

평가 수치와 손실을 한눈에 보려면 맨 아래 **3. 학습 결과 대시보드**를 실행하세요.
이미 저장된 학습 기록을 읽으므로 재학습은 필요 없습니다.

## 0. 공통 준비 — 두 모델에서 함께 사용

아래의 환경 탐색, import, 설정, 데이터 준비는 **경로 탐색 → import → 공통 함수 → 설정 → 데이터 준비 코드 셀**입니다.
YOLO26s만 실험할 때도 이 준비 구간은 먼저 실행하고, 실험 1 구간을 건너뛰어 실험 2로 이동하세요.

### 0-1. 저장소 경로와 실행 환경

저장소를 찾아 import 경로를 등록하고 `WANDB_MODE=disabled`를 설정합니다.
이 셀에서는 모델을 생성하거나 학습하지 않습니다.


In [ ]:
# YOLO11s 알약 검출 학습 → bbox 자르기 → 사전학습 EasyOCR 문자 인식
# OCR 자체를 학습하거나 YOLO와 공동 학습하는 코드는 아닙니다.
# 환경 준비: 저장소 루트에서 uv sync 실행 후 해당 환경을 커널로 선택합니다.
# 참고: https://docs.ultralytics.com/models/yolo11/
# 참고: https://www.jaided.ai/easyocr/documentation/
import os
import random
import subprocess
import sys
from pathlib import Path

# 자동 탐색에 실패하면 저장소를 내려받은 실제 경로를 지정하세요.
# 예: os.environ["PILL_BBOX_ROOT"] = str(Path.home() / "projects" / "pill-bbox")
configured_root = os.environ.get("PILL_BBOX_ROOT")
current_dir = Path.cwd().resolve()
if configured_root:
    candidates = [Path(configured_root).expanduser().resolve()]
else:
    candidates = []
    for parent in (current_dir, *current_dir.parents):
        candidates.extend([parent, parent / "pill-bbox"])
    candidates.append(Path.home() / "projects" / "pill-bbox")
root = next(
    (candidate for candidate in candidates if (candidate / "src" / "config.py").is_file()),
    None,
)
# Colab에 노트북만 올렸다면 공통 설정을 가져올 저장소를 준비합니다.
if root is None and "google.colab" in sys.modules and not configured_root:
    clone_dir = current_dir / "pill-bbox"
    if clone_dir.exists():
        raise RuntimeError(
            f"{clone_dir} 폴더에 src/config.py가 없습니다. "
            "PILL_BBOX_ROOT에 올바른 저장소 경로를 지정하세요."
        )
    print("Colab에 pill-bbox 저장소를 내려받습니다.")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/def-uk/pill-bbox.git", str(clone_dir)],
        check=True,
    )
    if (clone_dir / "src" / "config.py").is_file():
        root = clone_dir
if root is None:
    raise RuntimeError(
        f"저장소를 찾지 못했습니다. 현재 커널 경로: {current_dir}\n"
        "먼저 실행 환경에 pill-bbox 저장소를 clone하거나 업로드하세요.\n"
        '그다음 os.environ["PILL_BBOX_ROOT"]에 src/config.py가 있는 '
        "저장소 폴더의 실제 경로를 지정하고 이 셀을 다시 실행하세요."
    )
print(f"저장소 경로: {root}")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ["WANDB_MODE"] = "disabled"

### 0-2. 패키지와 공통 추론·분할 함수 준비

YOLO, EasyOCR, 이미지 처리 도구를 불러온 뒤 다음 코드 셀에서 공통 추론·분할 함수를 정의합니다.
모두 노트북 안에서 실행되므로 별도 `src/ocr_pipeline.py`나 메모리 모듈 등록 코드는 필요 없습니다.
이 구간은 함수 준비이며 학습 단계가 아닙니다. 두 코드 셀을 순서대로 실행하세요.


In [ ]:
# 현재 노트북 커널에서 필수 패키지를 확인하고 누락된 패키지만 설치합니다.
import importlib
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path
from urllib.request import urlopen

# 내부 상수 import 오류가 난 환경에서는 True로 한 번 실행합니다.
# 재설치 후 런타임을 재시작하고 False로 되돌린 뒤 처음부터 실행하세요.
REPAIR_ULTRALYTICS = False
ULTRALYTICS_REQUIREMENT = "ultralytics==8.4.182"

required_packages = {
    "gdown": "gdown",
    "torch": "torch",
    "torchvision": "torchvision",
    "cv2": "opencv-python-headless",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    # 최종 검증·그래프에서 뒤늦게 불러오는 패키지도 학습 전에 확인합니다.
    "scipy": "scipy",
    "seaborn": "seaborn",
    "polars": "polars",
    "yaml": "pyyaml",
    "easyocr": "easyocr",
    "ultralytics": ULTRALYTICS_REQUIREMENT,
    "wandb": "wandb",
    "dotenv": "python-dotenv",
    "IPython": "ipython",
}
# 잘린 traceback 대신 오류의 원인과 전체 호출 경로를 표시합니다.
sys.tracebacklimit = None
print(f"현재 커널 Python: {sys.executable}")
missing_packages = []
for module_name, package_name in required_packages.items():
    available = importlib.util.find_spec(module_name) is not None
    print(f"{module_name}: {'설치됨' if available else '설치 필요'}")
    if not available:
        missing_packages.append(package_name)

if missing_packages or REPAIR_ULTRALYTICS:
    uv_command = shutil.which("uv")
    if uv_command is None:
        # 공식 설치 스크립트로 uv를 준비합니다.
        with urlopen("https://astral.sh/uv/install.sh", timeout=60) as response:
            installer = response.read().decode("utf-8")
        subprocess.run(["sh"], input=installer, text=True, check=True)
        uv_command = str(Path.home() / ".local" / "bin" / "uv")
    if REPAIR_ULTRALYTICS:
        # torch/CUDA 환경은 유지하고 Ultralytics 파일만 같은 버전으로 다시 설치합니다.
        subprocess.run(
            [
                uv_command,
                "pip",
                "install",
                "--python",
                sys.executable,
                "--reinstall-package",
                "ultralytics",
                "--no-deps",
                "--no-cache",
                ULTRALYTICS_REQUIREMENT,
            ],
            check=True,
        )
    if missing_packages:
        subprocess.run(
            [uv_command, "pip", "install", "--python", sys.executable, *missing_packages],
            check=True,
        )
    importlib.invalidate_caches()
    # invalidate_caches만으로는 sys.modules에 남은 이전 버전이 교체되지 않습니다.
    raise RuntimeError(
        "패키지 설치 완료. 여기서 실행을 멈추세요. "
        "Colab의 런타임 > 세션 다시 시작을 실행한 뒤 "
        "REPAIR_ULTRALYTICS = False로 바꾸고 첫 셀부터 실행하세요. "
        "학습 셀만 바로 재실행하면 이전 모듈과 새 파일이 섞일 수 있습니다."
    )

import_errors = []
for module_name in required_packages:
    try:
        imported_module = importlib.import_module(module_name)
        print(
            f"{module_name}: import 성공 ({getattr(imported_module, '__version__', '버전 미표시')})"
        )
    except Exception:
        import traceback

        import_errors.append(f"{module_name}:\n{traceback.format_exc()}")
if import_errors:
    raise RuntimeError(
        "패키지 import 검사 실패:\n"
        + "\n".join(import_errors)
        + "\n설치 직후라면 런타임을 재시작한 뒤 첫 셀부터 다시 실행하세요."
    )
print("필수 패키지 import 검사 완료. 다음 셀을 실행하세요.")

In [ ]:
from typing import Any

import cv2
import easyocr
import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from ultralytics import YOLO
from ultralytics.engine.results import Results

from src.config import PROJECT_ROOT, get_device

In [ ]:
# 별도 파일 없이 두 YOLO 실험에서 함께 사용하는 OCR 추론 함수를 정의합니다.
def select_inference_image(image_path: Path | None, search_dirs: tuple[Path, ...]) -> Path:
    """지정한 이미지를 확인하거나 우선순위에 따라 폴더에서 이미지를 찾는다."""
    if image_path is not None:
        if not image_path.is_file():
            raise FileNotFoundError(
                f"지정한 추론 이미지가 없습니다: {image_path}\n"
                "설정 셀에서 이미지 경로를 실제 파일 또는 None으로 바꾸세요."
            )
        return image_path
    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    for search_dir in search_dirs:
        candidate = next(
            (
                path
                for path in search_dir.rglob("*")
                if path.suffix.lower() in extensions and path.is_file()
            ),
            None,
        )
        if candidate is not None:
            print(f"추론 이미지 자동 선택 (탐색 폴더: {search_dir}): {candidate}")
            return candidate
    raise FileNotFoundError(
        "추론할 이미지가 없습니다. 확인한 폴더: "
        + ", ".join(str(path) for path in search_dirs)
        + "\n테스트 이미지를 업로드하거나 실제 이미지 경로를 지정하세요."
    )


def read_crop_text(
    crop: np.ndarray,
    reader: easyocr.Reader,
    confidence: float,
    min_crop_size: int,
    rotation_angles: tuple[int, ...],
) -> list[dict[str, Any]]:
    """확대한 알약 crop의 원본·대비 보정본 중 평균 신뢰도가 높은 OCR 결과를 반환한다."""
    scale = max(1.0, min_crop_size / min(crop.shape[:2]))
    enlarged = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(enlarged, cv2.COLOR_BGR2GRAY)
    enhanced = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    candidates = []
    for view in (enlarged, enhanced):
        words = reader.readtext(view, detail=1, rotation_info=list(rotation_angles))
        candidates.append([word for word in words if word[2] >= confidence])
    words = max(candidates, key=lambda items: np.mean([word[2] for word in items]) if items else 0)
    return [
        {
            "text": word[1],
            "confidence": float(word[2]),
            "bbox_crop_scaled": np.asarray(word[0]).tolist(),
        }
        for word in words
    ]


def predict_with_ocr(
    detector: YOLO,
    reader: easyocr.Reader,
    image_path: Path,
    device: str,
    detection_confidence: float = 0.25,
    ocr_confidence: float = 0.3,
    crop_padding: float = 0.05,
    min_crop_size: int = 160,
    rotation_angles: tuple[int, ...] = (90, 180, 270),
) -> tuple[Results, list[dict[str, Any]]]:
    """YOLO 검출 영역을 잘라 OCR를 적용하고 검출 결과와 알약별 기록을 반환한다."""
    if not 0 <= detection_confidence <= 1 or not 0 <= ocr_confidence <= 1:
        raise ValueError("검출·OCR 신뢰도는 0~1이어야 합니다.")
    if crop_padding < 0 or min_crop_size <= 0:
        raise ValueError("crop 여백은 0 이상, 최소 크기는 양수여야 합니다.")
    if not image_path.is_file():
        raise FileNotFoundError(f"지정한 추론 이미지가 없습니다: {image_path}")
    img = cv2.imread(str(image_path))
    if img is None:
        raise ValueError(f"이미지를 읽을 수 없습니다: {image_path}")
    prediction = detector.predict(
        source=img, conf=detection_confidence, device=device, verbose=False
    )[0]
    records = []
    height, width = img.shape[:2]
    for box in prediction.boxes:
        left, top, right, bottom = box.xyxy[0].cpu().tolist()
        pad_x = (right - left) * crop_padding
        pad_y = (bottom - top) * crop_padding
        x1, y1 = max(0, int(left - pad_x)), max(0, int(top - pad_y))
        x2 = min(width, int(np.ceil(right + pad_x)))
        y2 = min(height, int(np.ceil(bottom + pad_y)))
        if x2 <= x1 or y2 <= y1:
            continue
        words = read_crop_text(
            img[y1:y2, x1:x2], reader, ocr_confidence, min_crop_size, rotation_angles
        )
        class_id = int(box.cls.item())
        records.append(
            {
                "class_id": class_id,
                "class_name": prediction.names[class_id],
                "detection_confidence": float(box.conf.item()),
                "bbox_xyxy": [left, top, right, bottom],
                "crop_xyxy": [x1, y1, x2, y2],
                "ocr_text": " ".join(word["text"] for word in words),
                "ocr_words": words,
            }
        )
    return prediction, records


def select_validation_groups(
    group_classes: dict[str, set[int]],
    val_fraction: float,
    seed: int,
    preferred_val_groups: set[str] | None = None,
) -> set[str]:
    """검증 클래스를 학습에서도 볼 수 있도록 알약 조합 단위의 검증 그룹을 고른다."""
    from collections import Counter

    if not 0 < val_fraction < 1:
        raise ValueError("검증 비율은 0보다 크고 1보다 작아야 합니다.")
    if len(group_classes) < 2:
        raise ValueError("독립된 알약 조합이 2개 이상 필요합니다.")
    remaining = Counter(class_id for classes in group_classes.values() for class_id in classes)
    candidates = sorted(group_classes)
    random.Random(seed).shuffle(candidates)
    if preferred_val_groups is not None:
        # 기존 검증 그룹을 우선 선택해 가능한 범위에서 원래 분할을 유지합니다.
        candidates.sort(key=lambda group: group not in preferred_val_groups)
    target = min(len(candidates) - 1, max(1, round(len(candidates) * val_fraction)))
    selected = set()
    for group in candidates:
        classes = group_classes[group]
        # 이 조합을 빼면 학습에서 사라지는 클래스가 있으면 학습에 남깁니다.
        if not classes or any(remaining[class_id] <= 1 for class_id in classes):
            continue
        selected.add(group)
        remaining.subtract(classes)
        if len(selected) == target:
            break
    if not selected:
        raise ValueError(
            "조합 분리와 학습 클래스 확보를 동시에 만족하는 검증 조합이 없습니다. "
            "각 클래스가 다른 조합에도 등장하도록 데이터를 추가해야 합니다. "
            "같은 조합의 사진을 양쪽에 나누거나 클래스를 제거해 해결하지 않습니다."
        )
    if len(selected) < target:
        print(f"클래스 확보를 위해 검증 조합을 목표 {target}개 대신 {len(selected)}개 사용합니다.")
    return selected


def train_with_error_log(detector: YOLO, data_yaml: Path, config: dict[str, Any]) -> Any:
    """학습 중 ImportError의 전체 traceback을 저장하고 원인 메시지를 표시한다."""
    import traceback
    from datetime import datetime

    try:
        return detector.train(data=str(data_yaml.resolve()), **config)
    except ImportError as error:
        # 화면 출력 제한과 무관하게 원본 traceback과 연결된 예외를 보존합니다.
        log_dir = PROJECT_ROOT / "runs" / "errors"
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f"train_import_error_{datetime.now():%Y%m%d_%H%M%S_%f}.txt"
        log_path.write_text(traceback.format_exc(), encoding="utf-8")
        raise RuntimeError(
            f"학습 또는 최종 검증 중 ImportError: {error}\n"
            f"전체 오류 기록: {log_path}\n"
            "이 파일 내용을 확인하세요. 저장된 best.pt·last.pt가 있다면 보존하세요."
        ) from None

### 0-3. 데이터 경로·seed·기본 학습 설정

- `DATA_YAML`: 두 실험이 공통으로 사용할 데이터셋 설정입니다.
- `IMAGE_PATH`: 추론할 이미지입니다. `None`이면 테스트 폴더에서 찾고, 없으면 생성된 검증 폴더에서 찾습니다.
- `SEED = 42`: 데이터 분할과 실험 재현을 위한 값입니다.
- `TRAIN_CONFIG`: 기본 학습 조건입니다. 실행 이름은 YOLO11s이며, YOLO26s는 뒤에서 독립적인 본 학습 설정을 사용합니다.
- `SMOKE_TEST = False`: **YOLO11s 실험**의 기본값은 전체 데이터·20 epoch입니다.
  True로 바꾸면 1 epoch 확인용 설정이 적용됩니다.
  YOLO26s에는 뒤의 `YOLO26_SMOKE_TEST`가 적용됩니다.

설정만 정의하며 여기서는 학습하지 않습니다.


In [ ]:
# 준비된 YOLO 데이터셋의 YAML 경로를 지정하세요.
# images/train, images/val과 대응하는 labels/train, labels/val이 필요합니다.
# 라벨 한 줄: class_id center_x center_y width height (좌표는 0~1)
# train/val에는 같은 원본 이미지의 복사본이 섞이지 않도록 분리하세요.
DATA_YAML = PROJECT_ROOT / "data" / "processed" / "yolo" / "data.yaml"
# None이면 테스트 폴더, 없으면 이 노트북의 YOLO 출력 폴더 내 val에서 선택합니다.
# 특정 이미지를 쓰려면 실제 파일 경로를 지정하세요.
IMAGE_PATH: Path | None = None
SEED = 42
DEVICE = get_device()
TRAIN_CONFIG = {
    "epochs": 20,
    "batch": 8,
    "imgsz": 640,
    "patience": 10,
    "lr0": 0.001,
    "seed": SEED,
    "deterministic": True,
    "workers": 0,
    "project": str(PROJECT_ROOT / "runs" / "detect"),
    "name": "JJAM99_yolo11s",
    "device": DEVICE,
    # 각인 방향을 보존하도록 반전 증강을 끕니다.
    "fliplr": 0.0,
    "flipud": 0.0,
}
# 기본은 20 epoch 본 학습입니다. 짧은 동작 확인이 필요할 때만 True로 바꾸세요.
SMOKE_TEST = False
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"device={DEVICE}, seed={SEED}")

### 0-4. 공통 데이터 준비 — JSON → YOLO 라벨, train/val 분할

`data.yaml`이 없으면 원본 annotation을 변환하고, 같은 알약 조합이 train과 val에
섞이지 않도록 그룹 단위로 나눕니다. 유효하지 않은 크기의 bbox가 있는 이미지는 전체 제외하고 기록합니다.
기존 `data.yaml`이 있으면 해당 데이터셋을 재사용합니다.

**데이터 전처리 단계이며 모델 학습은 아닙니다.** 두 실험은 같은 데이터 분할을 사용합니다.

검증으로 옮겨도 모든 검증 클래스가 학습에 남는 조합만 선택합니다. 한 조합에만 있는 클래스는 학습에 둡니다.
기존 data.yaml은 재생성하지 않으며, 필요하면 YOLO26s의 2-2 점검에서 기존 분할을 보완합니다.


In [ ]:
# 데이터 준비 및 확인: data.yaml이 없으면 이 셀에서 원본 JSON을 변환합니다.
# 원본이 없으면 기존 다운로드 함수로 데이터를 준비합니다.
import json
import math
import shutil
from collections import defaultdict

ANNOTATION_DIR = PROJECT_ROOT / "data" / "raw" / "train_annotations"
TRAIN_IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "train_images"
VAL_FRACTION = 0.2
# 같은 알약 조합의 다른 촬영 사진도 같은 분할에 둡니다.
# 독립된 조합이 하나뿐이면 데이터 조합을 추가해야 합니다.
if not DATA_YAML.is_file():
    if not any(ANNOTATION_DIR.rglob("*.json")) or not TRAIN_IMAGE_DIR.is_dir():
        from getpass import getpass

        from dotenv import load_dotenv

        from data_process.download_data import download_data

        # 저장소 밖에 업로드한 .env도 읽되 기존 환경 변수는 유지합니다.
        load_dotenv(Path.cwd() / ".env")
        if "google.colab" in sys.modules:
            load_dotenv(Path("/content") / ".env")
            if not os.environ.get("DATA_URL"):
                from google.colab import userdata

                try:
                    data_url = userdata.get("DATA_URL")
                except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                    data_url = None
                if data_url:
                    os.environ["DATA_URL"] = data_url
        if not os.environ.get("DATA_URL"):
            data_url = getpass("데이터 ZIP의 Google Drive 공유 링크(DATA_URL): ").strip()
            if not data_url:
                raise ValueError("DATA_URL이 비어 있습니다. 데이터 ZIP 공유 링크를 입력하세요.")
            os.environ["DATA_URL"] = data_url
        download_data(output_dir=PROJECT_ROOT / "data" / "raw")
    annotation_paths = sorted(ANNOTATION_DIR.rglob("*.json"))
    if not annotation_paths or not TRAIN_IMAGE_DIR.is_dir():
        raw_dir = PROJECT_ROOT / "data" / "raw"
        available_dirs = (
            sorted(path.name for path in raw_dir.iterdir() if path.is_dir())
            if raw_dir.is_dir()
            else []
        )
        raise FileNotFoundError(
            f"YOLO 변환에 필요한 원본 데이터가 없습니다.\n"
            f"실행 환경 저장소: {PROJECT_ROOT}\n"
            f"JSON 폴더: {ANNOTATION_DIR} (JSON {len(annotation_paths)}개)\n"
            f"이미지 폴더: {TRAIN_IMAGE_DIR} (존재: {TRAIN_IMAGE_DIR.is_dir()})\n"
            f"data/raw 하위 폴더: {available_dirs}\n"
            "Colab의 파일 목록에서 위 경로에 실제 데이터가 있는지 확인하세요. "
            "로컬 PC의 데이터는 Colab에 자동 전송되지 않습니다. "
            "colab.ipynb의 데이터 받기를 실행하거나 이 셀의 원본 경로를 수정하세요."
        )
    image_metadata = {}
    image_labels = defaultdict(set)
    category_names = {}
    invalid_bbox_records = []
    excluded_images = set()
    for annotation_path in annotation_paths:
        with annotation_path.open(encoding="utf-8") as file:
            content = json.load(file)
        local_images = {item["id"]: item for item in content["images"]}
        for category in content["categories"]:
            category_id = int(category["id"])
            name = str(category.get("name") or category_id)
            if category_id in category_names and category_names[category_id] != name:
                raise ValueError(f"클래스 이름 충돌: {category_id}")
            category_names[category_id] = name
        for label in content["annotations"]:
            if label.get("ignore", 0) or label.get("iscrowd", 0):
                continue
            image = local_images[label["image_id"]]
            image_name = image["file_name"]
            relative_path = Path(image_name)
            if relative_path.is_absolute() or ".." in relative_path.parts:
                raise ValueError(f"잘못된 이미지 상대 경로: {image_name}")
            width, height = int(image["width"]), int(image["height"])
            if width <= 0 or height <= 0:
                raise ValueError(f"잘못된 이미지 크기: {image_name}")
            if image_name in image_metadata and image_metadata[image_name] != (width, height):
                raise ValueError(f"이미지 크기 충돌: {image_name}")
            image_metadata[image_name] = (width, height)
            left, top, box_width, box_height = map(float, label["bbox"])
            if not all(math.isfinite(v) for v in (left, top, box_width, box_height)):
                raise ValueError(f"유효하지 않은 bbox: {annotation_path}")
            right, bottom = min(width, left + box_width), min(height, top + box_height)
            left, top = max(0.0, left), max(0.0, top)
            if right <= left or bottom <= top:
                # 일부 라벨만 제거하면 누락된 알약을 배경으로 학습하므로 이미지 전체를 제외합니다.
                excluded_images.add(image_name)
                invalid_bbox_records.append(
                    {
                        "annotation_path": str(annotation_path.relative_to(PROJECT_ROOT)),
                        "annotation_id": label.get("id"),
                        "image_name": image_name,
                        "image_size": [width, height],
                        "bbox": label["bbox"],
                        "reason": "non_positive_bbox_after_clipping",
                    }
                )
                continue
            image_labels[image_name].add(
                (
                    int(label["category_id"]),
                    (left + right) / (2 * width),
                    (top + bottom) / (2 * height),
                    (right - left) / width,
                    (bottom - top) / height,
                )
            )
    if invalid_bbox_records:
        report_dir = PROJECT_ROOT / "data" / "processed" / "correction_records"
        report_dir.mkdir(parents=True, exist_ok=True)
        report_path = report_dir / "JJAM99_invalid_bboxes.json"
        report_path.write_text(
            json.dumps(invalid_bbox_records, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        print(
            f"크기가 없는 bbox {len(invalid_bbox_records)}개 발견, "
            f"이미지 {len(excluded_images)}개 전체 제외. 기록: {report_path}"
        )
    for image_name in excluded_images:
        image_labels.pop(image_name, None)
    if not image_labels:
        raise ValueError("변환할 유효한 라벨이 없습니다.")
    class_ids = sorted({label[0] for labels in image_labels.values() for label in labels})
    class_mapping = {category_id: index for index, category_id in enumerate(class_ids)}
    groups = defaultdict(list)
    for image_name in sorted(image_labels):
        groups[Path(image_name).stem.split("_", 1)[0]].append(image_name)
    group_names = sorted(groups)
    if len(group_names) < 2:
        raise ValueError("독립된 알약 조합이 2개 이상 있어야 train/val로 분리할 수 있습니다.")
    group_classes = {
        group: {label[0] for image_name in image_names for label in image_labels[image_name]}
        for group, image_names in groups.items()
    }
    val_groups = select_validation_groups(group_classes, VAL_FRACTION, SEED)
    # 쓰기 전에 누락 이미지와 출력 파일명 충돌을 확인합니다.
    output_names = set()
    for image_name in image_labels:
        if not (TRAIN_IMAGE_DIR / image_name).is_file():
            raise FileNotFoundError(f"연결 이미지 누락: {image_name}")
        output_name = Path(image_name).with_suffix(".txt").as_posix()
        if output_name in output_names:
            raise ValueError(f"라벨 파일명 충돌: {output_name}")
        output_names.add(output_name)
    output_dir = DATA_YAML.parent
    if output_dir.exists() and any(output_dir.iterdir()):
        raise FileExistsError(
            f"미완성 또는 기존 출력이 있습니다. DATA_YAML을 새 출력 폴더로 지정하세요: {output_dir}"
        )
    counts = {"train": 0, "val": 0}
    split_classes = {"train": set(), "val": set()}
    for group_name, image_names in groups.items():
        split = "val" if group_name in val_groups else "train"
        for image_name in image_names:
            destination = output_dir / "images" / split / image_name
            label_path = output_dir / "labels" / split / Path(image_name).with_suffix(".txt")
            destination.parent.mkdir(parents=True, exist_ok=True)
            label_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(TRAIN_IMAGE_DIR / image_name, destination)
            lines = []
            for category_id, center_x, center_y, box_width, box_height in sorted(
                image_labels[image_name]
            ):
                split_classes[split].add(category_id)
                lines.append(
                    f"{class_mapping[category_id]} {center_x:.8f} {center_y:.8f} "
                    f"{box_width:.8f} {box_height:.8f}"
                )
            label_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
            counts[split] += 1
    prepared_config = {
        "path": str(output_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "names": {class_mapping[key]: category_names.get(key, str(key)) for key in class_ids},
    }
    with DATA_YAML.open("w", encoding="utf-8") as file:
        yaml.safe_dump(prepared_config, file, allow_unicode=True, sort_keys=False)
    manifest = {
        "seed": SEED,
        "val_fraction": VAL_FRACTION,
        "split_strategy": "group_holdout_with_train_class_coverage",
        "group_by": "filename_before_underscore",
        "train_groups": sorted(set(groups) - val_groups),
        "val_groups": sorted(val_groups),
        "original_category_to_class": class_mapping,
        "image_counts": counts,
        "excluded_images": sorted(excluded_images),
        "invalid_bbox_count": len(invalid_bbox_records),
    }
    (output_dir / "split_manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    unseen = split_classes["val"] - split_classes["train"]
    print(f"생성 완료: {counts}, 클래스 {len(class_ids)}개, seed={SEED}")
    if unseen:
        print(f"주의: 검증셋에만 있는 클래스 {sorted(unseen)} — 분할/데이터 보완이 필요합니다.")
else:
    print(f"기존 데이터셋 사용: {DATA_YAML}")

# 생성된 데이터셋 설정을 확인합니다.
if not DATA_YAML.is_file():
    raise FileNotFoundError(f"YOLO 데이터셋을 준비하고 DATA_YAML을 설정하세요: {DATA_YAML}")
with DATA_YAML.open(encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)
if not isinstance(dataset_config, dict):
    raise ValueError("data.yaml은 매핑 형식이어야 합니다.")
for key in ("train", "val", "names"):
    if not dataset_config.get(key):
        raise ValueError(f"data.yaml에 {key} 항목이 필요합니다.")
print(dataset_config)

## 1. 실험 1 — YOLO11s 학습 + EasyOCR 각인 추론

실험 흐름: **YOLO11s 학습 → 검출 평가 → EasyOCR 준비 → 이미지 추론**.
여기부터 실험 2 제목 직전까지는 모두 YOLO11s 실험입니다.

### 1-1. YOLO11s 생성 및 학습

`yolo11s.pt` 사전학습 가중치를 시작점으로 알약 bbox와 클래스를 학습합니다.
**아래 `model.train(...)`이 YOLO11s를 실제로 학습시키는 코드입니다.**

| 설정 | 학습 범위 | 실행 이름 |
|---|---|---|
| `SMOKE_TEST = True` | 학습 데이터 5%, 1 epoch | `JJAM99_yolo11s_smoke` |
| `SMOKE_TEST = False` | 전체 학습 데이터, 기본 최대 20 epoch | `JJAM99_yolo11s` |

학습 결과는 `runs/detect/`의 해당 실행 폴더에 저장하며,
이후 평가·추론에는 `best_weights`에 기록된 `best.pt`를 사용합니다.
이미 실행 폴더가 있으면 Ultralytics가 실행 이름에 번호를 붙일 수 있습니다.


In [ ]:
# 최초 실행에는 YOLO11s 사전학습 가중치를 다운로드합니다.
# 모델은 runs 아래 저장하여 저장소 루트에 가중치가 생기지 않게 합니다.
weights_dir = PROJECT_ROOT / "runs" / "pretrained"
weights_dir.mkdir(parents=True, exist_ok=True)
model = YOLO(str(weights_dir / "yolo11s.pt"))
train_config = dict(TRAIN_CONFIG)
if SMOKE_TEST:
    train_config.update(epochs=1, fraction=0.05, name="JJAM99_yolo11s_smoke")
# fraction은 학습 데이터에만 적용됩니다. 검증셋이 크면 검증에 시간이 걸립니다.
print(f"학습 조건: {train_config}")
train_results = train_with_error_log(model, DATA_YAML, train_config)
best_weights = Path(model.trainer.best)
if not best_weights.is_file():
    raise FileNotFoundError(f"학습된 가중치를 찾을 수 없습니다: {best_weights}")
print(f"학습 가중치: {best_weights}")

### 1-2. YOLO11s 검출 성능 평가 — 추가 학습 없음

학습한 `best.pt`를 `detector`로 불러와 검증셋에서 mAP50과 mAP50-95를 측정합니다.
이 수치는 **알약 위치·클래스 검출 성능**이며 OCR 문자 인식 정확도가 아닙니다.


In [ ]:
# 평가에는 학습에 사용하지 않은 val 분할을 사용합니다.
detector = YOLO(str(best_weights))
metrics = detector.val(
    data=str(DATA_YAML.resolve()),
    split="val",
    device=DEVICE,
    imgsz=TRAIN_CONFIG["imgsz"],
    batch=TRAIN_CONFIG["batch"],
    workers=0,
)
print({"mAP50": metrics.box.map50, "mAP50-95": metrics.box.map})
# 마지막 비교표에 사용할 평가 결과와 가중치 경로를 함께 기록합니다.
yolo11_evaluation = {
    "weights": str(best_weights.resolve()),
    "mAP50 (%)": float(metrics.box.map50) * 100,
    "mAP50-95 (%)": float(metrics.box.map) * 100,
    "Precision (%)": float(metrics.box.mp) * 100,
    "Recall (%)": float(metrics.box.mr) * 100,
}

### 1-3. YOLO11s에 연결할 EasyOCR 준비 — OCR 학습 없음

한국어·영어 사전학습 EasyOCR Reader를 준비합니다. 기본 장치는 CPU입니다.
검출·문자 인식 신뢰도 기준과 crop 여백을 지정하며, 모델을 학습하지 않습니다.
OCR 가중치가 실행 환경에 없으면 Reader가 최초 실행 시 다운로드할 수 있습니다.


In [ ]:
# OCR 가중치는 최초 실행 시 다운로드하며 여러 이미지에 같은 Reader를 재사용합니다.
# 기본은 CPU입니다. GPU를 쓰려면 OCR_DEVICE = DEVICE로 바꾸세요.
OCR_DEVICE = "cpu"
OCR_LANGUAGES = ["ko", "en"]
reader = easyocr.Reader(
    OCR_LANGUAGES,
    gpu=False if OCR_DEVICE == "cpu" else OCR_DEVICE,
    model_storage_directory=str(PROJECT_ROOT / "runs" / "easyocr"),
)
DETECTION_CONFIDENCE = 0.25
OCR_CONFIDENCE = 0.3
CROP_PADDING = 0.05

### 1-4. YOLO11s + EasyOCR 이미지 추론 및 출력 — 추가 학습 없음

학습한 YOLO11s로 알약을 검출하고, bbox 주변을 잘라 확대·대비 보정한 뒤 EasyOCR로 각인을 읽습니다.

- 이미지: **YOLO11s**의 bbox·클래스·검출 신뢰도입니다.
- `records`: 알약별 검출 정보와 `ocr_text`, 문자별 신뢰도입니다.
- `ocr_text`가 빈 문자열이면 설정한 신뢰도 기준을 넘은 OCR 문자가 없다는 뜻입니다.

bbox가 잘 표시되는지와 각인이 실제 문자와 맞게 읽혔는지는 각각 확인해야 합니다.
이 셀까지가 **실험 1의 끝**입니다.


In [ ]:
# 단일 이미지 추론: OCR 문자 좌표는 확대된 crop 기준으로 반환됩니다.
image_path = select_inference_image(
    IMAGE_PATH,
    (PROJECT_ROOT / "data" / "raw" / "test_images", DATA_YAML.parent / "images" / "val"),
)
print(f"추론 이미지: {image_path}")
prediction, records = predict_with_ocr(
    detector,
    reader,
    image_path,
    device=DEVICE,
    detection_confidence=DETECTION_CONFIDENCE,
    ocr_confidence=OCR_CONFIDENCE,
    crop_padding=CROP_PADDING,
)
print(records)
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(prediction.plot(), cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()
# OCR 문자열은 각인 인식 결과이며 약품 종류 확정값은 아닙니다.
# OCR 성능 평가는 별도의 실제 각인 정답과 비교해야 합니다.

## 2. 실험 2 — YOLO26s 본 학습 + EasyOCR 각인 추론

실행 순서: **2-1 설정 → 2-2 데이터 점검 → 2-3 학습 → 2-4 평가 → 2-5 검출 진단 → 2-6 OCR**.
YOLO11s 학습은 필요 없습니다. 공통 준비 구간은 먼저 실행하세요.

### 2-1. YOLO26s 독립 학습 설정

이전의 학습 데이터 5%·1 epoch 설정을 기본값으로 사용하지 않습니다.
`YOLO26_SMOKE_TEST = False`가 기본이며 전체 train 데이터로 최대 20 epoch 학습합니다.

| 설정 | 기본값과 목적 |
|---|---|
| 학습량 | `fraction=1.0`, `epochs=100`, `patience=20`으로 전체 데이터 학습 및 조기 종료 |
| 최적화 | `AdamW`, `lr0=0.001`, warmup·cosine 학습률 스케줄 |
| 입력·메모리 | `imgsz=640`, `batch=4`, 디스크/메모리 이미지 캐시 비활성화 |
| 증강 | mosaic 0.5, 작은 회전·크기·색 변화, 반전·mixup 비활성화 |
| 정밀도 | `amp=False`로 혼합 정밀도 및 별도 AMP 점검 가중치 다운로드 없이 기준 실험 |
| 기록 | epoch별 검증, 학습 곡선, `best.pt`/`last.pt` 저장 |
| 실행 이름 | `JJAM99_yolo26s_full`로 기존 smoke 실행과 구분 |

이 설정은 알약 데이터의 학습 기준값이며 성능 최적값으로 검증된 것은 아닙니다.
실제 결과를 mAP와 손실로 확인한 뒤 조정하세요. batch·imgsz도 실행 환경에 맞게 바꿀 수 있습니다.
기존 `yolo26s.pt`를 사용하며, 경로가 다르면 `YOLO26_PRETRAINED_WEIGHTS`만 지정하세요.
사전학습 가중치를 찾지 못하면 새로 다운로드하지 않고 경로를 안내합니다.

같은 데이터 분할·seed를 사용하지만 YOLO11s 기본 설정과 학습량·증강·최적화 조건은 다릅니다.
참고: [Ultralytics 미세조정 안내](https://docs.ultralytics.com/guides/finetuning-guide/).


In [ ]:
# 공통 준비를 실행한 뒤 이 셀부터 실행하세요. YOLO11s 학습은 필수가 아닙니다.
YOLO26_DATA_YAML = DATA_YAML
YOLO26_IMAGE_PATH = IMAGE_PATH
YOLO26_DEVICE = get_device()
YOLO26_SEED = SEED
# 조합 분리를 유지하며 모든 검증 클래스를 학습에서도 볼 수 있도록 보완합니다.
YOLO26_REPAIR_SPLIT = True
YOLO26_VAL_FRACTION = 0.2
YOLO26_PRETRAINED_WEIGHTS = PROJECT_ROOT / "runs" / "pretrained" / "yolo26s.pt"
# 기본은 본 학습입니다. True는 동작 확인용 1 epoch이며 성능 판단에 쓰지 않습니다.
YOLO26_SMOKE_TEST = False
YOLO26_TRAIN_CONFIG = {
    "epochs": 20,
    "fraction": 1.0,
    "batch": 4,
    "imgsz": 640,
    "patience": 20,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_bias_lr": 0.0,
    "cos_lr": True,
    "mosaic": 0.5,
    "close_mosaic": 10,
    "mixup": 0.0,
    "degrees": 10.0,
    "translate": 0.1,
    "scale": 0.2,
    "hsv_h": 0.01,
    "hsv_s": 0.2,
    "hsv_v": 0.2,
    "fliplr": 0.0,
    "flipud": 0.0,
    "seed": YOLO26_SEED,
    "deterministic": True,
    "workers": 0,
    "cache": False,
    # 혼합 정밀도와 별도 AMP 점검 가중치 다운로드 없이 기준 실험을 진행합니다.
    "amp": False,
    "val": True,
    "plots": True,
    "save": True,
    "resume": False,
    "project": str(PROJECT_ROOT / "runs" / "detect"),
    "name": "JJAM99_yolo26s_full",
    "device": YOLO26_DEVICE,
}
YOLO26_OCR_LANGUAGES = ["ko", "en"]
YOLO26_OCR_DEVICE = "cpu"
YOLO26_OCR_CONFIG = {
    "detection_confidence": 0.25,
    "ocr_confidence": 0.3,
    "crop_padding": 0.05,
    "min_crop_size": 160,
    "rotation_angles": (90, 180, 270),
}
# 설정을 바꾸면 데이터 점검과 검증 결과도 다시 계산해야 합니다.
yolo26_data_audit = None
yolo26_evaluation = None
if not YOLO26_DATA_YAML.is_file():
    raise FileNotFoundError("먼저 공통 데이터 준비 셀을 실행해 data.yaml을 생성하세요.")
print(f"YOLO26s: device={YOLO26_DEVICE}, seed={YOLO26_SEED}")
print(
    "1 epoch 확인용"
    if YOLO26_SMOKE_TEST
    else f"본 학습: 전체 train 데이터, 최대 {YOLO26_TRAIN_CONFIG['epochs']} epoch"
)
print(f"데이터: {YOLO26_DATA_YAML}")

### 2-2. 데이터 점검·클래스 확보를 반영한 분할·정답 bbox 확인

준비된 YOLO 라벨을 점검하고, **검증에만 있는 클래스**나 **양쪽에 걸친 알약 조합**이 있으면
같은 조합 전체를 한쪽에 배정하면서 모든 검증 클래스를 학습에서도 볼 수 있도록 분할을 보완합니다.
한 조합에만 있는 클래스는 학습에 남깁니다. 따라서 일부 클래스는 검증 지표에 포함되지 않을 수 있습니다.
검증 크기는 전체 조합의 약 20%를 목표로 하며, 클래스 확보 조건에 따라 줄어들 수 있습니다.

- `YOLO26_REPAIR_SPLIT=True`가 기본입니다. 보완이 필요 없으면 기존 분할을 그대로 사용합니다.
- 기존 이미지·라벨은 유지하고 `data/processed/yolo_splits/<계획 ID>/`에
  `train.txt`, `val.txt`, `data.yaml`, `split_manifest.json`만 저장합니다. 별도 코드 파일은 필요 없습니다.
- 클래스 ID·이름은 유지하며, 이후 YOLO26s 학습·평가에는 보완된 `YOLO26_DATA_YAML`을 사용합니다.
- 잘못된 bbox·누락 라벨·동일 이미지 경로 중복은 자동 수정하지 않고 중단합니다.
- 조합 분리와 클래스 확보를 동시에 만족할 검증 조합이 없으면 추가 데이터가 필요하다고 안내합니다.

새 분할에서 예전 모델의 mAP와 직접 비교하지 마세요. 모델끼리 비교하려면 같은 새 분할로 둘 다 재학습해야 합니다.
표에 나온 클래스·이미지 수와 분할별 정답 bbox 샘플을 확인한 뒤 2-3 학습 셀을 실행하세요.


In [ ]:
import hashlib
import json

# 원본 이미지·JSON은 수정하지 않고, 준비된 YOLO 이미지 경로와 라벨을 점검합니다.
from collections import Counter, defaultdict

import pandas as pd
from IPython.display import display
from ultralytics.data.utils import IMG_FORMATS, img2label_paths
from ultralytics.utils import DATASETS_DIR

# 중간에 점검이 실패하면 이전의 통과 상태를 재사용하지 않습니다.
yolo26_data_audit = None
with YOLO26_DATA_YAML.open(encoding="utf-8") as file:
    yolo26_dataset_config = yaml.safe_load(file)
if not isinstance(yolo26_dataset_config, dict):
    raise ValueError("data.yaml은 매핑 형식이어야 합니다.")
yolo26_names = yolo26_dataset_config.get("names")
if isinstance(yolo26_names, list):
    yolo26_names = dict(enumerate(yolo26_names))
elif isinstance(yolo26_names, dict):
    yolo26_names = {int(key): value for key, value in yolo26_names.items()}
else:
    raise ValueError("data.yaml에 클래스 names가 필요합니다.")
if not yolo26_names or set(yolo26_names) != set(range(len(yolo26_names))):
    raise ValueError("names의 클래스 ID는 0부터 연속된 정수여야 합니다.")
yolo26_dataset_root = Path(yolo26_dataset_config.get("path") or YOLO26_DATA_YAML.parent)
# Ultralytics와 동일하게 상대 path가 현재 경로에 없으면 datasets 디렉터리를 기준으로 봅니다.
if not yolo26_dataset_root.is_absolute() and not yolo26_dataset_root.exists():
    yolo26_dataset_root = DATASETS_DIR / yolo26_dataset_root
yolo26_dataset_root = yolo26_dataset_root.resolve()
yolo26_split_images = {}
yolo26_class_counts = {}
yolo26_image_class_counts = {}
yolo26_audit_rows = []
yolo26_errors = []
for split in ("train", "val"):
    split_value = yolo26_dataset_config.get(split)
    if not split_value:
        raise ValueError(f"data.yaml에 {split} 경로가 필요합니다.")
    split_entries = split_value if isinstance(split_value, list) else [split_value]
    image_paths = []
    for entry in split_entries:
        split_path = (yolo26_dataset_root / entry).resolve()
        if split_path.is_dir():
            image_paths.extend(
                path.resolve()
                for path in split_path.rglob("*")
                if path.is_file() and path.suffix[1:].lower() in IMG_FORMATS
            )
        elif split_path.is_file() and split_path.suffix.lower() == ".txt":
            # ./로 시작하는 목록 항목은 Ultralytics와 동일하게 목록 파일 기준입니다.
            for line in split_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line:
                    candidate = split_path.parent / line if line.startswith("./") else Path(line)
                    image_paths.append(candidate.resolve())
        else:
            raise FileNotFoundError(f"{split} 이미지 폴더/목록을 찾지 못했습니다: {split_path}")
    image_paths = sorted(set(image_paths))
    if not image_paths:
        raise ValueError(f"{split} 이미지가 없습니다.")
    yolo26_split_images[split] = image_paths
    class_counts = Counter()
    missing_labels = 0
    empty_labels = 0
    invalid_rows = 0
    for image_path, label_name in zip(image_paths, img2label_paths(image_paths), strict=True):
        label_path = Path(label_name)
        if not image_path.is_file():
            raise FileNotFoundError(f"목록에 연결된 이미지가 없습니다: {image_path}")
        if not label_path.is_file():
            missing_labels += 1
            if len(yolo26_errors) < 10:
                yolo26_errors.append(f"라벨 누락: {label_path}")
            continue
        image_counts = Counter()
        yolo26_image_class_counts[image_path] = image_counts
        label_lines = label_path.read_text(encoding="utf-8").splitlines()
        if not any(line.strip() for line in label_lines):
            empty_labels += 1
        for row_number, line in enumerate(label_lines, start=1):
            if not line.strip():
                continue
            try:
                values = [float(value) for value in line.split()]
                if len(values) != 5 or not all(math.isfinite(value) for value in values):
                    raise ValueError("유한한 숫자 5개가 필요합니다.")
                class_id, center_x, center_y, box_width, box_height = values
                if not class_id.is_integer() or int(class_id) not in yolo26_names:
                    raise ValueError("클래스 ID가 names 범위에 없습니다.")
                if not (0 <= center_x <= 1 and 0 <= center_y <= 1):
                    raise ValueError("bbox 중심 좌표는 0~1이어야 합니다.")
                if not (0 < box_width <= 1 and 0 < box_height <= 1):
                    raise ValueError("bbox 너비·높이는 0보다 크고 1 이하여야 합니다.")
                if (
                    center_x - box_width / 2 < -1e-5
                    or center_y - box_height / 2 < -1e-5
                    or center_x + box_width / 2 > 1 + 1e-5
                    or center_y + box_height / 2 > 1 + 1e-5
                ):
                    raise ValueError("bbox가 이미지 경계를 벗어납니다.")
                class_counts[int(class_id)] += 1
                image_counts[int(class_id)] += 1
            except ValueError as error:
                invalid_rows += 1
                if len(yolo26_errors) < 10:
                    yolo26_errors.append(f"{label_path}:{row_number}: {error}")
    yolo26_class_counts[split] = class_counts
    yolo26_audit_rows.append(
        {
            "분할": split,
            "이미지": len(image_paths),
            "유효 bbox": sum(class_counts.values()),
            "클래스": len(class_counts),
            "라벨 누락": missing_labels,
            "빈 라벨": empty_labels,
            "잘못된 행": invalid_rows,
        }
    )
display(pd.DataFrame(yolo26_audit_rows).set_index("분할"))
if yolo26_errors:
    raise ValueError("라벨을 먼저 확인하세요. 예시 최대 10개:\n" + "\n".join(yolo26_errors))
if any(not counts for counts in yolo26_class_counts.values()):
    raise ValueError("train과 val 모두에 유효한 bbox가 필요합니다.")
overlap = set(yolo26_split_images["train"]) & set(yolo26_split_images["val"])
if overlap:
    raise ValueError(f"train/val에 같은 이미지 경로가 {len(overlap)}개 있습니다.")
unseen = set(yolo26_class_counts["val"]) - set(yolo26_class_counts["train"])
group_overlap = {
    image_path.stem.split("_", 1)[0] for image_path in yolo26_split_images["train"]
} & {image_path.stem.split("_", 1)[0] for image_path in yolo26_split_images["val"]}
if unseen or group_overlap:
    print(f"검증셋 전용 클래스: {sorted(unseen)} / 양쪽에 있는 조합: {len(group_overlap)}개")
    if not YOLO26_REPAIR_SPLIT:
        raise ValueError(
            "분할 보완이 필요합니다. YOLO26_REPAIR_SPLIT=True로 점검 셀을 다시 실행하세요."
        )
    # 개별 이미지가 아니라 동일한 알약 조합 전체를 한쪽에만 배정합니다.
    grouped_paths = defaultdict(list)
    grouped_classes = defaultdict(set)
    for split_paths in yolo26_split_images.values():
        for image_path in split_paths:
            group = image_path.stem.split("_", 1)[0]
            grouped_paths[group].append(image_path)
            grouped_classes[group].update(yolo26_image_class_counts[image_path])
    previous_val_groups = {
        image_path.stem.split("_", 1)[0] for image_path in yolo26_split_images["val"]
    }
    repaired_val_groups = select_validation_groups(
        dict(grouped_classes), YOLO26_VAL_FRACTION, YOLO26_SEED, previous_val_groups
    )
    repaired_paths = {"train": [], "val": []}
    for group, image_paths in sorted(grouped_paths.items()):
        split = "val" if group in repaired_val_groups else "train"
        repaired_paths[split].extend(sorted(image_paths))
    repaired_counts = {}
    for split, image_paths in repaired_paths.items():
        counts = Counter()
        for image_path in image_paths:
            counts.update(yolo26_image_class_counts[image_path])
        repaired_counts[split] = counts
    if set(repaired_counts["val"]) - set(repaired_counts["train"]):
        raise RuntimeError("분할 후에도 학습에 없는 검증 클래스가 있습니다.")
    if any(not counts for counts in repaired_counts.values()):
        raise ValueError("분할 후 train과 val 모두에 유효한 bbox가 필요합니다.")
    # 실제 이미지·라벨을 이동하지 않고 절대 경로 목록으로 새 분할을 표현합니다.
    source_yaml = YOLO26_DATA_YAML.resolve()
    split_plan = {
        "source_yaml": str(source_yaml),
        "seed": YOLO26_SEED,
        "group_by": "filename_before_underscore",
        "requested_val_fraction": YOLO26_VAL_FRACTION,
        "train_groups": sorted(set(grouped_paths) - repaired_val_groups),
        "val_groups": sorted(repaired_val_groups),
        "previous_val_only_classes": sorted(unseen),
        "paths": {
            split: [str(image_path) for image_path in image_paths]
            for split, image_paths in repaired_paths.items()
        },
    }
    plan_text = json.dumps(split_plan, ensure_ascii=False, sort_keys=True)
    plan_id = hashlib.sha256(plan_text.encode("utf-8")).hexdigest()[:12]
    split_dir = PROJECT_ROOT / "data" / "processed" / "yolo_splits" / plan_id
    split_dir.mkdir(parents=True, exist_ok=True)
    repaired_config = {
        "path": str(yolo26_dataset_root),
        "names": yolo26_names,
    }
    for split, image_paths in repaired_paths.items():
        list_path = split_dir / f"{split}.txt"
        list_text = "\n".join(str(image_path) for image_path in image_paths) + "\n"
        if not list_path.is_file() or list_path.read_text(encoding="utf-8") != list_text:
            list_path.write_text(list_text, encoding="utf-8")
        repaired_config[split] = str(list_path.resolve())
    corrected_yaml = split_dir / "data.yaml"
    manifest_path = split_dir / "split_manifest.json"
    config_text = yaml.safe_dump(repaired_config, allow_unicode=True, sort_keys=False)
    if not corrected_yaml.is_file() or corrected_yaml.read_text(encoding="utf-8") != config_text:
        corrected_yaml.write_text(config_text, encoding="utf-8")
    manifest_text = json.dumps(split_plan, ensure_ascii=False, indent=2)
    if not manifest_path.is_file() or manifest_path.read_text(encoding="utf-8") != manifest_text:
        manifest_path.write_text(manifest_text, encoding="utf-8")
    YOLO26_DATA_YAML = corrected_yaml
    yolo26_dataset_config = repaired_config
    yolo26_split_images = repaired_paths
    yolo26_class_counts = repaired_counts
    yolo26_evaluation = None
    print(f"클래스 확보를 반영한 분할: {YOLO26_DATA_YAML}")
    display(
        pd.DataFrame(
            [
                {
                    "분할": split,
                    "이미지": len(repaired_paths[split]),
                    "유효 bbox": sum(counts.values()),
                    "클래스": len(counts),
                }
                for split, counts in repaired_counts.items()
            ]
        ).set_index("분할")
    )
    print("분할이 바뀌었습니다. 기존 모델의 mAP와 직접 비교하지 말고 새 분할로 재학습하세요.")
train_only = set(yolo26_class_counts["train"]) - set(yolo26_class_counts["val"])
if train_only:
    print(
        f"안내: train에만 있는 클래스 {len(train_only)}개 "
        f"(최대 20개 표시: {sorted(train_only)[:20]})는 이번 val에서 평가되지 않습니다."
    )
rare_classes = {key: count for key, count in yolo26_class_counts["train"].items() if count < 10}
if rare_classes:
    print(f"안내: train bbox가 10개 미만인 클래스 {len(rare_classes)}개. 데이터 보완을 권합니다.")
yolo26_data_audit = {
    "data_yaml": str(YOLO26_DATA_YAML.resolve()),
    "train_images": len(yolo26_split_images["train"]),
    "val_images": len(yolo26_split_images["val"]),
    "train_classes": sorted(yolo26_class_counts["train"]),
}
print("데이터 점검 통과. 아래 정답 bbox가 실제 알약을 감싸는지 확인하세요.")
# 이미지 픽셀은 분할별 최대 2장만 읽어 정답 bbox를 표시합니다.
figure, axes = plt.subplots(2, 2, figsize=(14, 9), squeeze=False)
for split, row_axes in zip(("train", "val"), axes, strict=True):
    samples = random.Random(YOLO26_SEED).sample(
        yolo26_split_images[split], min(2, len(yolo26_split_images[split]))
    )
    for axis, sample_path in zip(row_axes, samples, strict=False):
        sample_img = cv2.imread(str(sample_path))
        if sample_img is None:
            yolo26_data_audit = None
            raise ValueError(f"샘플 이미지를 읽지 못했습니다: {sample_path}")
        height, width = sample_img.shape[:2]
        label_path = Path(img2label_paths([sample_path])[0])
        for line in label_path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            class_id, center_x, center_y, box_width, box_height = map(float, line.split())
            x1, y1 = (
                int((center_x - box_width / 2) * width),
                int((center_y - box_height / 2) * height),
            )
            x2, y2 = (
                int((center_x + box_width / 2) * width),
                int((center_y + box_height / 2) * height),
            )
            cv2.rectangle(sample_img, (x1, y1), (x2, y2), (0, 200, 0), 3)
            cv2.putText(sample_img, str(int(class_id)), (x1, max(15, y1)), 0, 0.7, (0, 200, 0), 2)
        axis.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
        axis.set_title(f"{split} ground truth | {sample_path.name}")
    for axis in row_axes:
        axis.axis("off")
figure.tight_layout()
plt.show()

### 2-3. YOLO26s 본 학습 — 이 셀을 실행해야 새 가중치가 만들어집니다

데이터 점검을 통과한 뒤 `yolo26s.pt`에서 새 학습을 시작합니다.
기본 사전학습 가중치가 없으면 자동 다운로드합니다.
기존 smoke 실험을 재개하지 않으며 결과는 별도 학습 실행 폴더에 저장됩니다.
**아래 `yolo26_model.train(...)`이 실제 학습 코드입니다.**

- 기본: 전체 train 데이터로 최대 20 epoch, 검증 결과에 따른 조기 종료 patience 20.
- `YOLO26_SMOKE_TEST = True`로 바꾼 경우만 1 epoch. 이때도 전체 train 데이터를 사용합니다.
- 학습 후 `results.png`가 있으면 학습 곡선을 즉시 표시합니다.
- 이후 평가·추론은 이번 실행의 `yolo26_best_weights`를 사용합니다.

기존 smoke 결과는 수정된 설정만 다시 실행해도 좋아지지 않습니다.
**이 학습 셀부터 실제로 새로 학습한 뒤 평가·추론해야 합니다.** EasyOCR는 학습하지 않습니다.


In [ ]:
from IPython.display import Image, display

# 점검한 데이터로 사전학습 가중치에서 새 학습을 시작합니다. 이전 smoke 학습은 재개하지 않습니다.
if not yolo26_data_audit or yolo26_data_audit["data_yaml"] != str(YOLO26_DATA_YAML.resolve()):
    raise RuntimeError("YOLO26s 데이터 점검 셀을 먼저 실행하세요.")
# 기본 가중치는 없으면 Ultralytics가 다운로드합니다. 사용자 지정 경로는 확인합니다.
YOLO26_PRETRAINED_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
if not YOLO26_PRETRAINED_WEIGHTS.is_file():
    if YOLO26_PRETRAINED_WEIGHTS.name != "yolo26s.pt":
        raise FileNotFoundError(f"사용자 지정 가중치가 없습니다: {YOLO26_PRETRAINED_WEIGHTS}")
    print(f"YOLO26s 사전학습 가중치를 다운로드합니다: {YOLO26_PRETRAINED_WEIGHTS}")
# 이전 실험 모델을 CPU로 옮겨 GPU 메모리를 확보합니다.
for variable_name in ("model", "detector", "yolo26_model", "yolo26_detector"):
    previous_model = globals().get(variable_name)
    if previous_model is not None:
        previous_model.to("cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
random.seed(YOLO26_SEED)
np.random.seed(YOLO26_SEED)
torch.manual_seed(YOLO26_SEED)
yolo26_train_config = dict(YOLO26_TRAIN_CONFIG)
if YOLO26_SMOKE_TEST:
    # 클래스 누락을 피하도록 동작 확인에서도 전체 train 데이터를 사용합니다.
    yolo26_train_config.update(
        epochs=1, fraction=1.0, warmup_epochs=0.0, close_mosaic=0, name="JJAM99_yolo26s_smoke"
    )
elif yolo26_train_config["epochs"] <= 1 or yolo26_train_config["fraction"] != 1.0:
    raise ValueError("본 학습에는 epochs > 1, fraction=1.0을 지정하세요.")
yolo26_evaluation = None
yolo26_model = YOLO(str(YOLO26_PRETRAINED_WEIGHTS))
print(f"학습 조건: {yolo26_train_config}")
yolo26_train_results = train_with_error_log(yolo26_model, YOLO26_DATA_YAML, yolo26_train_config)
yolo26_best_weights = Path(yolo26_model.trainer.best)
if not yolo26_best_weights.is_file():
    raise FileNotFoundError(f"YOLO26s 학습 가중치를 찾을 수 없습니다: {yolo26_best_weights}")
print(f"YOLO26s 학습 완료: {yolo26_best_weights}")
# Ultralytics가 저장한 학습 곡선을 그대로 표시합니다.

yolo26_curve_path = yolo26_best_weights.parent.parent / "results.png"
if yolo26_curve_path.is_file():
    display(Image(filename=str(yolo26_curve_path)))
print("다음 평가 셀에서 best.pt의 mAP를 확인하세요.")

### 2-4. YOLO26s 검출 성능 평가 — 추가 학습 없음

학습한 `best.pt`를 `yolo26_detector`로 불러와 같은 검증셋에서 mAP50과 mAP50-95를 측정합니다.
`yolo26_metric_summary`에 수치를 저장합니다.

YOLO11s의 검증 수치와 비교할 수 있습니다. 비교할 때는 양쪽의 확인용/본 학습 설정과
epoch·데이터 사용 비율·최적화·증강 조건이 같은지 확인하세요. OCR 정확도는 이 수치로 평가하지 않습니다.


In [ ]:
# 검증: mAP는 알약 검출 성능입니다. 각인 정답이 없으므로 OCR 정확도는 평가하지 않습니다.
yolo26_detector = YOLO(str(yolo26_best_weights))
yolo26_metrics = yolo26_detector.val(
    data=str(YOLO26_DATA_YAML.resolve()),
    split="val",
    device=YOLO26_DEVICE,
    imgsz=YOLO26_TRAIN_CONFIG["imgsz"],
    batch=YOLO26_TRAIN_CONFIG["batch"],
    workers=0,
    plots=True,
    project=str(yolo26_best_weights.parent.parent),
    name="validation",
    exist_ok=True,
)
yolo26_metric_summary = {
    "mAP50": float(yolo26_metrics.box.map50),
    "mAP50-95": float(yolo26_metrics.box.map),
}
print({"model": "YOLO26s", **yolo26_metric_summary})
# 다른 실행의 학습 로그와 섞이지 않도록 평가한 가중치 경로를 함께 기록합니다.
yolo26_evaluation = {
    "weights": str(yolo26_best_weights.resolve()),
    "mAP50 (%)": float(yolo26_metrics.box.map50) * 100,
    "mAP50-95 (%)": float(yolo26_metrics.box.map) * 100,
    "Precision (%)": float(yolo26_metrics.box.mp) * 100,
    "Recall (%)": float(yolo26_metrics.box.mr) * 100,
}
if yolo26_metric_summary["mAP50"] == 0:
    print("검증 mAP50이 0입니다. 학습 성공으로 판단하지 말고 아래 진단 이미지와 손실을 확인하세요.")

### 2-5. 학습·검증 이미지에서 검출 진단 — OCR 전 확인

train과 val에서 한 장씩 골라 정상 신뢰도 기준과 낮은 진단용 기준의 예측을 나란히 보여줍니다.
이 단계는 **검출이 안 된 원인을 살펴보기 위한 추론**이며 학습이나 mAP 평가를 대체하지 않습니다.
낮은 기준에서 나타난 bbox는 오검출일 수 있으므로 검출 성공으로 간주하지 마세요.

- train·val 모두 검출 부족: 라벨 위치, 손실 변화, 실제 학습량을 먼저 확인합니다.
- train에서는 검출되지만 val에서 부족: 데이터 차이와 과적합을 살펴봅니다.
- val은 괜찮지만 특정 test만 부족: 이미지 크기·촬영 조건·클래스 구성을 확인합니다.

정량 성능과 손실 추세는 맨 아래 **3. 학습 결과 대시보드**에서 확인하세요.


In [ ]:
# OCR 없이 학습·검증 이미지에서 검출 자체가 되는지 먼저 확인합니다.
# 0.05는 낮은 점수의 예측이 있는지 보는 진단용이며, 정상 추론 기준 0.25를 바꾸지 않습니다.
YOLO26_DIAGNOSTIC_THRESHOLDS = (YOLO26_OCR_CONFIG["detection_confidence"], 0.05)
figure, axes = plt.subplots(2, 2, figsize=(15, 10), squeeze=False)
for split, row_axes in zip(("train", "val"), axes, strict=True):
    sample_path = random.Random(YOLO26_SEED).choice(yolo26_split_images[split])
    for axis, confidence in zip(row_axes, YOLO26_DIAGNOSTIC_THRESHOLDS, strict=True):
        result = yolo26_detector.predict(
            source=str(sample_path),
            conf=confidence,
            imgsz=YOLO26_TRAIN_CONFIG["imgsz"],
            device=YOLO26_DEVICE,
            save=False,
            verbose=False,
        )[0]
        count = len(result.boxes)
        axis.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
        axis.set_title(f"{split} | conf={confidence:.2f} | detections={count}")
        axis.axis("off")
        print(f"{split}: conf={confidence:.2f}, 검출 {count}개, {sample_path.name}")
figure.tight_layout()
plt.show()
print(
    "train에서도 검출이 없으면 정답 bbox·학습 손실·학습 epoch를 확인하세요. "
    "train에서는 되지만 val/test에서 안 되면 데이터 차이·과적합을 확인하세요."
)

### 2-6. YOLO26s + EasyOCR 이미지 추론·각인 표시·결과 저장 — 추가 학습 없음

학습한 YOLO26s로 알약을 검출하고, 사전학습 EasyOCR로 각인을 읽습니다.
이 셀에서도 **OCR 학습은 하지 않습니다.**

| 출력 | 내용 |
|---|---|
| 전체 이미지 | **YOLO26s**의 bbox·클래스·검출 신뢰도 |
| 알약별 crop 이미지 | 잘라낸 알약과 OCR가 읽은 문자. `(no text)`는 기준을 넘은 문자가 없다는 뜻 |
| `yolo26_records` | 알약별 bbox·클래스·검출 신뢰도·OCR 문자와 신뢰도 |
| `ocr/<이미지 이름>_ocr.json` | 이번 학습 실행 폴더 아래에 설정·검출 지표·OCR 결과 저장 |

이 셀에는 결과 JSON을 파일로 쓰는 코드가 포함되어 있습니다.
OCR Reader가 아직 없으면 준비하며, OCR 가중치가 실행 환경에 없을 때는 다운로드할 수 있습니다.

**실험 결과를 판단하는 기준:** 검출 성능은 검증 mAP로, 각인 인식은 실제 문자와의 비교로 확인합니다.
이미지 한 장의 출력이 정상인 것은 추론 동작 확인이며 전체 데이터 성능을 보장하지는 않습니다.

검출이 0개이면 OCR 결과도 비게 됩니다. 2-4 평가와 2-5 검출 진단을 먼저 확인하세요.


In [ ]:
# YOLO26s 검출 → 알약 crop → EasyOCR → bbox 이미지와 알약별 각인 출력
# 한 셀에서 Reader를 한 번 만들고 이미지마다 재사용합니다. 최초 실행에는 OCR 가중치를 받습니다.
if "yolo26_reader" not in globals():
    yolo26_reader = easyocr.Reader(
        YOLO26_OCR_LANGUAGES,
        gpu=False if YOLO26_OCR_DEVICE == "cpu" else YOLO26_OCR_DEVICE,
        model_storage_directory=str(PROJECT_ROOT / "runs" / "easyocr"),
    )
# OCR 언어·장치를 바꿨다면 del yolo26_reader 실행 후 이 셀을 다시 실행하세요.
if YOLO26_IMAGE_PATH is not None:
    yolo26_image_path = select_inference_image(YOLO26_IMAGE_PATH, ())
else:
    try:
        yolo26_image_path = select_inference_image(
            None, (PROJECT_ROOT / "data" / "raw" / "test_images",)
        )
    except FileNotFoundError:
        # 목록 파일을 쓰는 분할에서도 실제 검증 이미지 중에서 고릅니다.
        yolo26_image_path = random.Random(YOLO26_SEED).choice(yolo26_split_images["val"])
        print(f"검증 이미지 자동 선택: {yolo26_image_path}")
yolo26_prediction, yolo26_records = predict_with_ocr(
    yolo26_detector,
    yolo26_reader,
    yolo26_image_path,
    device=YOLO26_DEVICE,
    **YOLO26_OCR_CONFIG,
)
print(json.dumps(yolo26_records, ensure_ascii=False, indent=2))
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(yolo26_prediction.plot(), cv2.COLOR_BGR2RGB))
plt.title("YOLO26s detections")
plt.axis("off")
plt.show()
# crop 이미지 위에 각인 결과를 표시합니다.
# 빈 문자열은 OCR 신뢰도 기준을 넘은 문자가 없다는 뜻입니다.
for index, record in enumerate(yolo26_records, start=1):
    x1, y1, x2, y2 = record["crop_xyxy"]
    plt.figure(figsize=(5, 3))
    plt.imshow(cv2.cvtColor(yolo26_prediction.orig_img[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
    plt.title(f"#{index} OCR: {record['ocr_text'] or '(no text)'}")
    plt.axis("off")
    plt.show()
if not yolo26_records:
    print("검출 0개입니다. 2-4 평가와 2-5 train/val 진단 결과를 먼저 확인하세요.")
# 결과는 해당 학습 실행 폴더에 저장합니다. 원본 이미지·JSON은 수정하지 않습니다.
yolo26_output_dir = yolo26_best_weights.parent.parent / "ocr"
yolo26_output_dir.mkdir(parents=True, exist_ok=True)
yolo26_result_path = yolo26_output_dir / f"{yolo26_image_path.stem}_ocr.json"
yolo26_result_path.write_text(
    json.dumps(
        {
            "model": "YOLO26s",
            "weights": str(yolo26_best_weights),
            "image_path": str(yolo26_image_path),
            "data_yaml": str(YOLO26_DATA_YAML),
            "seed": YOLO26_SEED,
            "train_config": yolo26_train_config,
            "metrics": yolo26_metric_summary,
            "ocr_engine": "EasyOCR",
            "ocr_languages": YOLO26_OCR_LANGUAGES,
            "ocr_config": YOLO26_OCR_CONFIG,
            "records": yolo26_records,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"YOLO26s 검출 {len(yolo26_records)}개 / OCR 결과 저장: {yolo26_result_path}")

## 3. 학습 결과 대시보드 — 수치·그래프·모델 비교

**이미 학습했다면 학습 셀을 다시 실행할 필요가 없습니다.** 아래 두 코드 셀만 실행하세요.
OCR 셀도 실행할 필요 없습니다. 저장된 `results.csv`를 읽고 노트북에 표시하며,
추가 파일 생성·다운로드·학습은 하지 않습니다.

### 3-1. 모델별 학습 변화와 마지막 손실

각 모델의 **epoch별 mAP50·mAP50-95·Precision/Recall**, 그리고
**학습(train)·검증(val) 손실**을 그래프로 표시합니다. 그래프 아래에는 마지막 epoch의 손실 값을 표시합니다.

- 우선 현재 실행의 `best_weights` 또는 `yolo26_best_weights`로 학습 폴더를 찾습니다.
- 런타임을 다시 시작했다면 공통 준비에서 저장소 경로를 먼저 설정하세요.
  학습 폴더가 모델별로 하나만 있으면 자동 선택합니다.
- 여러 실행이 있으면 임의로 선택하지 않고 후보를 출력합니다.
  아래 `RESULTS_CSV_PATHS`에 보고 싶은 `results.csv`를 지정하고 다시 실행하세요.
- 모델 하나만 학습했어도 있는 결과부터 표시합니다.

| 보는 값 | 해석 |
|---|---|
| mAP50 / mAP50-95 | 높을수록 검출 성능이 좋습니다. mAP50-95는 bbox 위치를 더 엄격하게 평가합니다. |
| Precision | 검출했다고 예측한 것 중 정답인 비율입니다. |
| Recall | 실제 정답 중 검출해 낸 비율입니다. |
| `box_loss` | bbox 위치 관련 손실입니다. 같은 실행 안에서 추세를 봅니다. |
| `cls_loss` | 클래스 예측 관련 손실입니다. 같은 실행 안에서 추세를 봅니다. |
| `dfl_loss`, `l1_loss` 등 | 모델·버전에 따라 있는 위치 회귀 관련 손실입니다. 실제 기록된 항목만 표시합니다. |

mAP 그래프는 0~1로 표시하고, 마지막 비교표·막대그래프는 백분율(%)로 표시합니다.
예를 들어 mAP `0.70`은 비교표에서 `70.00%`입니다. 단순 분류 정확도와는 다른 지표입니다.

손실은 대체로 낮아지는 방향이 좋지만, **다른 구조의 모델 간 손실 절댓값으로 우열을 정하지 않습니다.**
학습 손실만 낮아지고 검증 손실은 올라가면서 mAP도 정체·하락한다면 과적합을 의심해 볼 수 있습니다.
1 epoch 결과는 점 하나만 있으므로 학습 추세나 과적합을 판단할 수 없습니다.

지표 설명 참고: [Ultralytics 성능 지표 안내](https://docs.ultralytics.com/guides/yolo-performance-metrics/).


In [ ]:
# 저장된 기록을 읽어 표시만 합니다. 모델 학습이나 결과 파일 쓰기는 없습니다.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import MaxNLocator

# None이면 현재 학습 경로 또는 모델별로 유일한 학습 폴더를 찾습니다.
# 예: "YOLO11s": PROJECT_ROOT / "runs" / "detect" / "JJAM99_yolo11s_smoke" / "results.csv"
RESULTS_CSV_PATHS = {"YOLO11s": None, "YOLO26s": None}
metric_columns = {
    "mAP50 (%)": "metrics/mAP50(B)",
    "mAP50-95 (%)": "metrics/mAP50-95(B)",
    "Precision (%)": "metrics/precision(B)",
    "Recall (%)": "metrics/recall(B)",
}
run_settings = {
    "YOLO11s": ("best_weights", "yolo11_evaluation", "JJAM99_yolo11s"),
    "YOLO26s": ("yolo26_best_weights", "yolo26_evaluation", "JJAM99_yolo26s"),
}
# 재실행할 때 이전 결과가 남지 않도록 집계를 초기화합니다.
evaluation_rows = []
loss_rows = []
training_histories = {}

for model_name, (weights_variable, evaluation_variable, run_prefix) in run_settings.items():
    selected_path = RESULTS_CSV_PATHS[model_name]
    weights_path = globals().get(weights_variable)
    if selected_path is None and weights_path is not None:
        selected_path = Path(weights_path).parent.parent / "results.csv"
    if selected_path is None:
        candidates = sorted((PROJECT_ROOT / "runs" / "detect").glob(f"{run_prefix}*/results.csv"))
        if len(candidates) == 1:
            selected_path = candidates[0]
        elif len(candidates) > 1:
            print(f"{model_name}: 실행 폴더가 여러 개입니다. RESULTS_CSV_PATHS에서 선택하세요.")
            for candidate in candidates:
                print(f"  {candidate}")
            continue
        else:
            print(f"{model_name}: 학습 기록이 없습니다. 학습한 CSV의 경로를 지정하세요.")
            continue
    csv_path = Path(selected_path)
    if not csv_path.is_file():
        print(f"{model_name}: results.csv가 없습니다: {csv_path}")
        continue
    try:
        history = pd.read_csv(csv_path)
    except (OSError, pd.errors.EmptyDataError, pd.errors.ParserError) as error:
        print(f"{model_name}: CSV를 읽을 수 없습니다: {error}")
        continue
    history.columns = history.columns.str.strip()
    if "epoch" not in history.columns:
        print(f"{model_name}: epoch 열이 없습니다: {csv_path}")
        continue
    loss_columns = [
        column
        for column in history.columns
        if column.startswith(("train/", "val/")) and column.endswith("_loss")
    ]
    for column in ["epoch", *metric_columns.values(), *loss_columns]:
        if column in history.columns:
            history[column] = pd.to_numeric(history[column], errors="coerce")
    history = history.replace([np.inf, -np.inf], np.nan)
    history = history.dropna(subset=["epoch"]).sort_values("epoch").reset_index(drop=True)
    if history.empty:
        print(f"{model_name}: 유효한 epoch 기록이 없습니다: {csv_path}")
        continue
    training_histories[model_name] = history
    print(f"\n{model_name} — 기록 {len(history)} epoch / {csv_path}")
    if len(history) == 1:
        print("1 epoch 동작 확인 결과입니다. 학습 추세나 최종 성능은 판단할 수 없습니다.")

    # 로그 최고 기록과 best.pt 재평가 결과는 출처를 구분합니다.
    score_column = metric_columns["mAP50-95 (%)"]
    score_values = history.get(score_column, pd.Series(dtype=float)).dropna()
    evaluation_row = None
    if not score_values.empty:
        peak = history.loc[score_values.idxmax()]
        evaluation_row = {
            "모델": model_name,
            "지표 출처": f"학습 로그 최고 epoch {int(peak['epoch'])}",
            "기록 epoch 수": len(history),
            **{
                label: float(peak.get(column, np.nan)) * 100
                for label, column in metric_columns.items()
            },
        }
    evaluation = globals().get(evaluation_variable)
    if isinstance(evaluation, dict) and "weights" in evaluation:
        evaluated_run = Path(evaluation["weights"]).resolve().parent.parent
        if evaluated_run == csv_path.resolve().parent:
            evaluation_row = {
                "모델": model_name,
                "지표 출처": "best.pt 재평가",
                "기록 epoch 수": len(history),
                **{label: evaluation.get(label, np.nan) for label in metric_columns},
            }
    if evaluation_row is not None:
        evaluation_rows.append(evaluation_row)

    loss_names = sorted({column.split("/", 1)[1] for column in loss_columns})
    panel_count = 3 + len(loss_names)
    figure, axes = plt.subplots(
        (panel_count + 1) // 2, 2, figsize=(14, 3.4 * ((panel_count + 1) // 2))
    )
    axes = axes.ravel()
    metric_panels = [
        ("mAP50", ["metrics/mAP50(B)"]),
        ("mAP50-95", ["metrics/mAP50-95(B)"]),
        ("Precision / Recall", ["metrics/precision(B)", "metrics/recall(B)"]),
    ]
    for axis, (title, columns) in zip(axes[:3], metric_panels, strict=True):
        for column in columns:
            if column in history and history[column].notna().any():
                axis.plot(history["epoch"], history[column], marker="o", label=column.split("/")[1])
        axis.set(title=title, xlabel="Epoch", ylabel="Score (0-1)", ylim=(0, 1.05))
        axis.grid(alpha=0.3)
        if axis.lines:
            axis.legend()
        else:
            axis.text(0.5, 0.5, "No metric recorded", transform=axis.transAxes, ha="center")
    for axis, loss_name in zip(axes[3:panel_count], loss_names, strict=True):
        for split in ("train", "val"):
            column = f"{split}/{loss_name}"
            if column in history and history[column].notna().any():
                axis.plot(history["epoch"], history[column], marker="o", label=split)
        axis.set(title=loss_name, xlabel="Epoch", ylabel="Loss")
        axis.grid(alpha=0.3)
        if axis.lines:
            axis.legend()
    for axis in axes[:panel_count]:
        axis.xaxis.set_major_locator(MaxNLocator(integer=True))
    for axis in axes[panel_count:]:
        axis.set_visible(False)
    figure.suptitle(f"{model_name} | training history ({len(history)} epochs)", fontsize=15)
    figure.tight_layout(rect=(0, 0, 1, 0.97))
    plt.show()
    last = history.iloc[-1]
    if loss_columns:
        loss_rows.append(
            {
                "모델": model_name,
                "마지막 epoch": int(last["epoch"]),
                **{column: last[column] for column in loss_columns},
            }
        )
    else:
        print(f"{model_name}: CSV에 학습·검증 손실 항목이 없습니다.")

if loss_rows:
    print("마지막 기록 epoch의 손실 (best.pt 재평가 손실과는 별도입니다)")
    loss_summary = pd.DataFrame(loss_rows).set_index("모델")
    display(loss_summary.round(4).fillna("—"))
elif training_histories:
    print("손실 표를 표시할 수 없습니다. CSV에 train/*_loss 또는 val/*_loss 열이 필요합니다.")
else:
    print("표시할 기록이 없습니다. RESULTS_CSV_PATHS와 학습 결과 위치를 확인하세요.")

### 3-2. YOLO11s vs YOLO26s — 평가 수치 비교와 결과 읽기

아래 표와 막대그래프는 두 모델의 검출 지표를 나란히 표시합니다. **높을수록 좋습니다.**

- 같은 학습 폴더의 평가 결과가 메모리에 있으면 **`best.pt 재평가`** 값을 사용합니다.
  위의 각 모델 평가 셀을 실행하면 이 값이 기록됩니다. 재학습은 필요 없습니다.
- 평가 기록이 없으면 **학습 로그에서 mAP50-95가 가장 높았던 epoch**의 지표를 함께 사용합니다.
  이것은 저장된 `best.pt`를 별도로 재평가한 값과 같다고 단정하지 않습니다.
- `지표 출처`가 다른 모델끼리는 동일한 기준으로 평가 셀을 실행한 뒤 비교하세요.
- 손실은 앞의 마지막 epoch 표에서 확인하며, mAP와 합쳐 종합 점수를 만들지 않습니다.

어느 모델이 이번 기록에서 더 높은 mAP50-95를 보였는지 차이를 출력합니다.
다만 **같은 데이터 분할·epoch·사용 비율·입력 크기·seed**에서 비교해야 해석할 수 있습니다.
1 epoch 확인용 결과만으로 최종 모델을 선정하지 마세요. OCR 인식 정확도는 이 비교에 포함되지 않습니다.


In [ ]:
# 위의 3-1 코드 셀을 먼저 실행하세요. 비교표와 그래프만 표시합니다.
if not globals().get("evaluation_rows"):
    print("먼저 3-1 셀에서 학습 기록을 불러오세요. 지표가 없는 CSV라면 mAP 열도 확인하세요.")
else:
    evaluation_summary = pd.DataFrame(evaluation_rows).set_index("모델")
    display(evaluation_summary.round(2).fillna("—"))
    metric_labels = list(metric_columns)
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for axis, labels, title in zip(
        axes,
        [metric_labels[:2], metric_labels[2:]],
        ["Detection mAP (higher is better)", "Precision / Recall (higher is better)"],
        strict=True,
    ):
        evaluation_summary[labels].plot.bar(ax=axis, rot=0, ylim=(0, 110))
        axis.set(title=title, xlabel="Model", ylabel="Score (%)")
        axis.grid(axis="y", alpha=0.3)
        for container, label in zip(axis.containers, labels, strict=True):
            value_labels = [
                f"{value:.2f}%" if pd.notna(value) else "N/A" for value in evaluation_summary[label]
            ]
            axis.bar_label(container, labels=value_labels, padding=3)
        axis.legend(loc="upper center")
    figure.tight_layout()
    plt.show()

    comparable = evaluation_summary.dropna(subset=["mAP50-95 (%)"])
    if len(comparable) == 2:
        sources = comparable["지표 출처"].str.startswith("best.pt")
        if sources.nunique() > 1:
            print("지표 출처가 다릅니다. 두 모델의 평가 셀을 실행한 뒤 다시 비교하세요.")
        else:
            ranked = comparable.sort_values("mAP50-95 (%)", ascending=False)
            difference = ranked.iloc[0]["mAP50-95 (%)"] - ranked.iloc[1]["mAP50-95 (%)"]
            if np.isclose(difference, 0):
                print("이번 비교표에서 두 모델의 mAP50-95는 같습니다.")
            else:
                print(
                    f"이번 비교표 기준: {ranked.index[0]}의 mAP50-95가 "
                    f"{ranked.index[1]}보다 {difference:.2f}%p 높습니다."
                )
            print("동일한 데이터·학습 조건인지 확인한 뒤 모델을 선택하세요.")
    elif len(comparable) == 1:
        print(
            "현재 한 모델의 결과만 있습니다. 다른 모델의 학습 기록도 불러오면 비교할 수 있습니다."
        )
    if (evaluation_summary["기록 epoch 수"] <= 1).any():
        print("1 epoch 결과가 포함되어 있습니다. 동작 확인 단계이므로 최종 우열은 판단하지 마세요.")